# Collect the results to send

Puts the small result files from your Drive into one zip, to send to Claude Code in one go:

- every file in `WORK/results` (the `.json` result files),
- from every run folder in `WORK/runs`: `history.json`, the predictions (`*_predictions.tsv`)
  and the sheets of misread words (`*_errors.png`),
- from `WORK/baseline`: the baseline's predictions.

Checkpoints (`.pt`), data sets and word lists are left out. The zip keeps the folder names
(`results/...`, `runs/<run>/...`), so every file says where it came from. It is saved on
Drive in `WORK/for_claude/` and, on a computer, also downloaded.

No GPU needed: a CPU runtime is enough (Runtime > Change runtime type). Run both cells.

In [ ]:
WORK = "/content/drive/MyDrive/meitei-word-recognition"   # the same folder as in the Phase 1 and 2 notebooks
FOLDERS = ["results", "runs", "baseline"]                   # what to collect, below WORK
KEEP = (".json", ".tsv", ".png", ".csv", ".txt")            # small text files and pictures only
MAX_MB = 20                                                 # larger files are left out

In [ ]:
import os, shutil, time, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
work = Path(WORK)
if not work.is_dir():
    raise SystemExit(f"{WORK} not found: change WORK in the first cell")

name = f"results_{time.strftime('%Y%m%d_%H%M')}.zip"
local = Path("/content") / name
count, size, left_out = 0, 0, []
with zipfile.ZipFile(local, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in FOLDERS:
        root = work / folder
        if not root.is_dir():
            print(f"(no {folder} folder)")
            continue
        for p in sorted(root.rglob("*")):
            if not p.is_file():
                continue
            if p.suffix.lower() not in KEEP or p.stat().st_size > MAX_MB * 1e6:
                left_out.append(p.relative_to(work))
                continue
            z.write(p, p.relative_to(work))
            count += 1
            size += p.stat().st_size

print(f"{count} files ({size / 1e6:.1f} MB before compression) in {name}")
print(f"left out: {len(left_out)} files (checkpoints and other large files)")
print()
print("What each run folder holds:")
for run in sorted(p for p in (work / "runs").iterdir() if p.is_dir()) if (work / "runs").is_dir() else []:
    have = [f for f in ("history.json", "val_predictions.tsv", "val_errors.png", "test_predictions.tsv")
            if (run / f).exists()]
    print(f"  {run.name}: {', '.join(have) if have else 'nothing to send yet'}")

target = work / "for_claude"
target.mkdir(exist_ok=True)
shutil.copy(local, target / name)
print()
print(f"Saved on Drive: {target / name}")
try:
    from google.colab import files
    files.download(str(local))            # on a computer the browser saves it too
except Exception:
    print("Download it from Drive (for_claude folder) and send it to Claude Code.")